# Symbolic Regression using Next-Generation Transformers

**GSoC 2025 – ML4SCI**

---

## Overview

This notebook implements **SymbolicGPT**, a transformer-based approach to symbolic regression that treats
equation discovery as a language-modelling problem.

### Architecture pipeline

```
Point Cloud  →  T-Net  →  order-invariant embedding
                                 ↓
              GPT Decoder  →  equation skeleton (constants masked as <C>)
                                 ↓
              BFGS  →  constant optimisation  →  final equation
```

### Key references

1. **SymbolicGPT** – Valipour et al., 2021 (arXiv 2106.14131)  
2. **Order-Invariant Embeddings + Sparse Decoding** – Malik et al., NeurIPS ML4PS 2025  
3. **Evolutionary + Transformer Hybrids (PIGP / SymbolicDPO)** – Jha et al., NeurIPS ML4PS 2024  

### Goals

* Learn a generalisable equation generator from thousands of synthetic examples  
* Evaluate on the **AI Feynman** benchmark  
* Report R², RMSE, token accuracy, and exact-match percentage  
* Study robustness to observation noise  

## 1. Install & Import Dependencies

In [ ]:
# ── Install (uncomment in Colab / fresh environment) ─────────────────────────
# !pip install torch sympy scipy numpy matplotlib tqdm requests

import os
import re
import csv
import io
import math
import copy
import time
import tarfile
import pathlib
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict

import sympy as sp
from sympy import symbols, lambdify, sympify, latex
from scipy.optimize import minimize
from scipy.stats import pearsonr

try:
    from sklearn.linear_model import LinearRegression
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)


## 2. Feynman Dataset Loading & Preprocessing

The **AI Feynman** symbolic regression benchmark (Udrescu & Tegmark, 2020) provides:

| File | Contents |
|---|---|
| `Feynman_with_units.tar.gz` | Numerical data files for Feynman equations (with units) |
| `Feynman_without_units.tar.gz` | Normalised data (dimensionless) |
| `Bonus_with_units.tar.gz` | Bonus equation data (with units) |
| `Bonus_without_units.tar.gz` | Bonus equation data (without units) |
| `FeynmanEquations.csv` | Equation metadata (name, formula, variable count) |
| `FeynmanEquationsDimensionless.csv` | Dimensionless equation metadata |
| `BonusEquations.csv` | Bonus equation metadata |

### Pipeline

1. **Extract** all `.tar.gz` archives in `data/`
2. **Load** equation metadata from CSV files
3. **Map** each equation name → extracted data file path
4. **Build** a unified DataFrame with columns `(name, formula, n_vars, data_path, kind)`

> **Note:** This notebook is configured to use local datasets from `data/` and does not rely on synthetic training data.


In [ ]:
# ── Feynman Dataset Loader (local files only) ────────────────────────────────

DATA_DIR = 'data'


def extract_tar_files():
    """Extract all .tar.gz archives inside DATA_DIR."""
    if not os.path.isdir(DATA_DIR):
        print(f"[DataLoader] Directory not found: {DATA_DIR}")
        return

    for file in sorted(os.listdir(DATA_DIR)):
        if file.endswith('.tar.gz'):
            tar_path = os.path.join(DATA_DIR, file)
            print(f"[DataLoader] Extracting {file}...")
            with tarfile.open(tar_path, 'r:gz') as tar:
                tar.extractall(DATA_DIR)


extract_tar_files()


def _read_csv_or_empty(path: str) -> pd.DataFrame:
    if os.path.exists(path):
        return pd.read_csv(path)
    print(f"[DataLoader] Missing CSV: {path}")
    return pd.DataFrame()


feynman_df = _read_csv_or_empty(os.path.join(DATA_DIR, 'FeynmanEquations.csv'))
bonus_df   = _read_csv_or_empty(os.path.join(DATA_DIR, 'BonusEquations.csv'))


def load_equation_data(file_path: str) -> pd.DataFrame:
    """Read equation point-cloud files with whitespace/comma delimiters."""
    return pd.read_csv(file_path, sep=r'\s+|,', engine='python', header=None)


def _normalise_name(path_or_name: str) -> str:
    base = os.path.basename(str(path_or_name)).strip().lower()
    for ext in ('.txt', '.dat', '.csv', '.tsv'):
        if base.endswith(ext):
            base = base[:-len(ext)]
    return base


def _row_field(row: pd.Series, candidates: list[str], default=''):
    for c in candidates:
        if c in row and pd.notna(row[c]):
            return str(row[c]).strip()
    return default


def _infer_dataset_type(file_path: str) -> str:
    lower = file_path.lower()
    if 'feynman_with_units' in lower:
        return 'Feynman_with_units'
    if 'feynman_without_units' in lower:
        return 'Feynman_without_units'
    if 'bonus' in lower:
        return 'Bonus'
    if 'dimensionless' in lower:
        return 'Dimensionless'
    return 'Other'


def _try_read_numeric_table(file_path: str) -> pd.DataFrame | None:
    """Try robust loading for txt/dat/csv numeric datasets."""
    ext = os.path.splitext(file_path)[1].lower()
    if ext in ('.txt', '.dat'):
        read_fns = [
            lambda p: pd.read_csv(p, sep=r'\s+|,', engine='python', header=None),
            lambda p: pd.read_csv(p, delim_whitespace=True, header=None),
        ]
    elif ext == '.csv':
        read_fns = [
            lambda p: pd.read_csv(p, header=None),
            lambda p: pd.read_csv(p, sep=r'\s+|,', engine='python', header=None),
        ]
    else:
        return None

    for fn in read_fns:
        try:
            df = fn(file_path)
            df = df.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all').dropna(axis=0, how='any')
            if df.shape[0] >= 3 and df.shape[1] >= 2:
                return df.reset_index(drop=True)
        except Exception:
            continue
    return None


def load_full_dataset(data_dir: str) -> list[tuple[str, pd.DataFrame]]:
    """Load all numeric datasets across Feynman_with_units/without_units/Bonus/Dimensionless."""
    datasets = []
    metadata_csv = {'feynmanequations', 'bonusequations'}

    for root, _, files in os.walk(data_dir):
        for file in sorted(files):
            lower = file.lower()
            stem = os.path.splitext(lower)[0]
            if stem in metadata_csv:
                continue
            if not lower.endswith(('.txt', '.dat', '.csv')):
                continue

            file_path = os.path.join(root, file)
            data = _try_read_numeric_table(file_path)
            if data is not None:
                datasets.append((file_path, data))

    return datasets


def _normalise_rows(df: pd.DataFrame, kind: str) -> list[dict]:
    records = []
    if df.empty:
        return records

    for _, row in df.iterrows():
        name = _row_field(row, ['Filename', 'filename', 'Name', 'name', 'Number', 'number'])
        formula = _row_field(row, ['Equation', 'Formula', 'formula', 'Output', 'output'])
        n_vars_raw = _row_field(row, ['# variables', '#variables', 'Num_Vars', 'num_vars', 'N'])

        n_vars = None
        try:
            n_vars = int(float(n_vars_raw))
        except (TypeError, ValueError):
            pass

        if name and formula:
            records.append({
                'name': name,
                'formula': formula,
                'n_vars': n_vars,
                'kind': kind,
            })
    return records


def build_dataset_catalogue(data_dir: str) -> tuple[list[dict], list[tuple[str, pd.DataFrame]]]:
    """Create robust equation↔dataset mapping across all dataset folders."""
    rows = _normalise_rows(feynman_df, 'feynman') + _normalise_rows(bonus_df, 'bonus')
    full_datasets = load_full_dataset(data_dir)

    by_norm_name: dict[str, list[dict]] = defaultdict(list)
    for file_path, data in full_datasets:
        by_norm_name[_normalise_name(file_path)].append({
            'data_path': file_path,
            'data': data,
            'dataset_type': _infer_dataset_type(file_path),
        })

    catalogue = []
    for row in rows:
        entry = dict(row)
        matches = by_norm_name.get(_normalise_name(row['name']), [])
        if matches:
            best = matches[0]
            entry['data_path'] = best['data_path']
            entry['dataset_type'] = best['dataset_type']
        else:
            entry['data_path'] = None
            entry['dataset_type'] = 'Missing'
        catalogue.append(entry)

    return catalogue, full_datasets


def load_equation_points(entry: dict,
                         n_points: int = 500,
                         rng: np.random.Generator | None = None) -> tuple[np.ndarray, np.ndarray] | tuple[None, None]:
    path = entry.get('data_path')
    if not path or not os.path.exists(path):
        return None, None

    try:
        df = load_equation_data(path)
    except Exception as exc:
        print(f"[DataLoader] Failed reading {path}: {exc}")
        return None, None

    data = df.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all').dropna(axis=0, how='any').to_numpy(dtype=np.float32)
    if data.ndim != 2 or data.shape[1] < 2:
        return None, None

    n_vars = entry.get('n_vars')
    if n_vars is None:
        n_vars = data.shape[1] - 1
    n_vars = int(n_vars)

    if data.shape[1] < n_vars + 1:
        return None, None

    X = data[:, :n_vars]
    y = data[:, n_vars]

    if n_points is not None and len(X) > n_points:
        if rng is None:
            rng = np.random.default_rng(42)
        idx = rng.choice(len(X), size=n_points, replace=False)
        X, y = X[idx], y[idx]

    return X.astype(np.float32), y.astype(np.float32)


def verify_equation(eq_str: str, data: pd.DataFrame) -> float | None:
    """Compute MSE between equation output and dataset targets when possible."""
    try:
        expr = sp.sympify(eq_str)
        variables = sorted(list(expr.free_symbols), key=lambda s: str(s))

        if len(variables) == 0:
            return None

        arr = data.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all').dropna(axis=0, how='any').to_numpy(dtype=np.float64)
        if arr.ndim != 2 or arr.shape[1] < len(variables) + 1:
            return None

        X = arr[:, :len(variables)]
        y_true = arr[:, -1]

        f = sp.lambdify(variables, expr, 'numpy')
        y_pred = f(*[X[:, i] for i in range(len(variables))])
        y_pred = np.asarray(y_pred, dtype=np.float64)
        if y_pred.ndim == 0:
            y_pred = np.full_like(y_true, y_pred, dtype=np.float64)

        if y_pred.shape[0] != y_true.shape[0]:
            return None
        if not np.all(np.isfinite(y_pred)):
            return None

        return float(np.mean((y_true - y_pred) ** 2))
    except Exception:
        return None


CATALOGUE, FULL_DATASETS = build_dataset_catalogue(DATA_DIR)

if FULL_DATASETS:
    stats = defaultdict(int)
    rows = []
    cols = []
    for fp, df in FULL_DATASETS:
        stats[_infer_dataset_type(fp)] += 1
        rows.append(df.shape[0])
        cols.append(df.shape[1])

    print(f"[DataLoader] Loaded numeric datasets: {len(FULL_DATASETS)}")
    print('[DataLoader] Dataset type counts:')
    for k in sorted(stats):
        print(f"  - {k:<22s}: {stats[k]}")
    print(f"[DataLoader] Row range: {min(rows)} .. {max(rows)}")
    print(f"[DataLoader] Col range: {min(cols)} .. {max(cols)}")
else:
    print('[DataLoader] No numeric datasets found under data/.')

mapped = [e for e in CATALOGUE if e.get('data_path')]
print(f"[DataLoader] Equation metadata rows: {len(CATALOGUE)}")
print(f"[DataLoader] Equation↔dataset mapped pairs: {len(mapped)}")

if mapped:
    print('\nSample equation ↔ dataset mappings:')
    for entry in mapped[:5]:
        print(
            f"  {entry['formula'][:55]:<55s} -> "
            f"{os.path.basename(entry['data_path']):<25s} [{entry['dataset_type']}]"
        )

    print('\nSample data-equation verification (MSE):')
    checked = 0
    for entry in mapped:
        table = _try_read_numeric_table(entry['data_path'])
        if table is None:
            continue
        err = verify_equation(entry['formula'], table)
        if err is not None:
            print(f"  {entry['name']:<14s} ({entry['dataset_type']:<22s}) -> MSE={err:.3e}")
            checked += 1
        if checked >= 3:
            break
else:
    print('[DataLoader] No valid equation↔dataset mappings found.')


## 3. Vocabulary & Tokenizer

Equations are represented as token sequences in **prefix (Polish) notation**.

### Why Prefix Notation?

| Property | Infix | Prefix |
|---|---|---|
| Parentheses needed? | Yes | **No** |
| Unambiguous parse? | Requires precedence rules | **Always** |
| Uniform operator arity | No | **Yes** |
| Suitable for LM training? | Messy | **Ideal** |

### Conversion Example

```
sin(x1) + x1²   →   add  sin  x1  pow  x1  <C>
```

### Token Types

| Type | Examples |
|---|---|
| Binary operators | `add sub mul div pow` |
| Unary operators | `sin cos tan exp log sqrt abs tanh` |
| Variables | `x1  x2  x3  x4  x5` |
| Constant placeholder | `<C>` (replaces every numeric literal) |
| Special | `<SOS>  <EOS>  <PAD>  <UNK>` |

Numerical constants in the training targets are replaced by `<C>`, forcing the
language model to learn equation *structure* rather than memorise values.  
A separate BFGS optimisation step (Section 6) recovers the actual constants.


In [ ]:
# ── Prefix-notation vocabulary ────────────────────────────────────────────────

PREFIX_BINARY_OPS = ['add', 'sub', 'mul', 'div', 'pow']
PREFIX_UNARY_OPS  = ['sin', 'cos', 'tan', 'exp', 'log', 'sqrt', 'abs', 'tanh']
PREFIX_OPS        = PREFIX_BINARY_OPS + PREFIX_UNARY_OPS
VARIABLES         = [f'x{i}' for i in range(1, 6)]      # x1 … x5
SPECIAL_TOKENS    = ['<C>', '<SOS>', '<EOS>', '<PAD>', '<UNK>']

ALL_TOKENS  = sorted(set(PREFIX_OPS + VARIABLES + SPECIAL_TOKENS))
VOCAB       = {tok: idx for idx, tok in enumerate(ALL_TOKENS)}
INV_VOCAB   = {idx: tok for tok, idx in VOCAB.items()}
VOCAB_SIZE  = len(VOCAB)

PAD_IDX = VOCAB['<PAD>']
SOS_IDX = VOCAB['<SOS>']
EOS_IDX = VOCAB['<EOS>']
UNK_IDX = VOCAB['<UNK>']

_VARIABLES_SET = set(VARIABLES)

print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'All tokens      : {ALL_TOKENS}')


# ── Sympy-expression → prefix token list ─────────────────────────────────────

def sympy_to_prefix(expr: sp.Expr) -> list:
    """Convert a SymPy expression tree into (possibly nested) prefix tokens.

    We first preserve tree structure (nested lists) and then flatten with
    `flatten_prefix` to obtain the final 1-D token sequence used by the model.
    """
    if isinstance(expr, sp.Symbol):
        name = str(expr)
        return [name] if name in _VARIABLES_SET else ['<UNK>']

    if isinstance(expr, sp.Number):
        return ['<C>']

    if isinstance(expr, sp.Add):
        args = list(sp.Add.make_args(expr))
        pos_terms, neg_terms = [], []
        for a in args:
            if a.could_extract_minus_sign():
                neg_terms.append(-a)
            else:
                pos_terms.append(a)

        if pos_terms and neg_terms:
            lhs = sp.Add(*pos_terms) if len(pos_terms) > 1 else pos_terms[0]
            rhs = sp.Add(*neg_terms) if len(neg_terms) > 1 else neg_terms[0]
            return ['sub', sympy_to_prefix(lhs), sympy_to_prefix(rhs)]

        result = sympy_to_prefix(args[0])
        for a in args[1:]:
            result = ['add', result, sympy_to_prefix(a)]
        return result

    if isinstance(expr, sp.Mul):
        args = list(sp.Mul.make_args(expr))
        num_parts, den_parts = [], []
        for a in args:
            if isinstance(a, sp.Pow) and a.exp.is_Number and float(a.exp) < 0:
                den_parts.append(sp.Pow(a.base, -a.exp))
            else:
                num_parts.append(a)

        if den_parts:
            num = sp.Mul(*num_parts) if len(num_parts) > 1 else (num_parts[0] if num_parts else sp.Integer(1))
            den = sp.Mul(*den_parts) if len(den_parts) > 1 else den_parts[0]
            return ['div', sympy_to_prefix(num), sympy_to_prefix(den)]

        result = sympy_to_prefix(args[0])
        for a in args[1:]:
            result = ['mul', result, sympy_to_prefix(a)]
        return result

    if isinstance(expr, sp.Pow):
        if expr.exp == sp.Rational(1, 2):
            return ['sqrt', sympy_to_prefix(expr.base)]
        return ['pow', sympy_to_prefix(expr.base), sympy_to_prefix(expr.exp)]

    unary_map = {
        sp.sin: 'sin', sp.cos: 'cos', sp.tan: 'tan',
        sp.exp: 'exp', sp.log: 'log', sp.sqrt: 'sqrt',
        sp.Abs: 'abs', sp.tanh: 'tanh',
    }
    func = getattr(expr, 'func', None)
    if func in unary_map and len(expr.args) == 1:
        return [unary_map[func], sympy_to_prefix(expr.args[0])]

    return ['<UNK>']


def flatten_prefix(tokens: list | str) -> list[str]:
    """Recursively flatten nested prefix token trees into a 1-D token list."""
    if isinstance(tokens, list):
        result = []
        for t in tokens:
            result.extend(flatten_prefix(t))
        return result
    else:
        return [tokens]


def tokenize_equation(eq_str: str) -> list[str]:
    """Parse an equation string and return flattened prefix tokens."""
    expr = sp.sympify(eq_str)
    tokens = sympy_to_prefix(expr)
    return flatten_prefix(tokens)


def equation_str_to_prefix(eq_str: str, n_vars: int = 5) -> list[str]:
    """
    Parse *eq_str* (infix string) via SymPy, then convert to flattened prefix tokens.
    """
    local_syms = {f'x{i}': sp.Symbol(f'x{i}') for i in range(1, n_vars + 1)}
    try:
        expr = sp.sympify(eq_str, locals=local_syms)
        tokens = sympy_to_prefix(expr)
        return flatten_prefix(tokens)
    except Exception:
        return ['<UNK>']


def tokens_to_ids(tokens: list[str]) -> list[int]:
    return [VOCAB.get(t, UNK_IDX) for t in tokens]


def ids_to_tokens(ids: list[int]) -> list[str]:
    return [INV_VOCAB.get(i, '<UNK>') for i in ids]


def prefix_tokens_to_infix(tokens: list[str]) -> str:
    """
    Reconstruct a human-readable infix expression from prefix tokens.
    (Used for display / debugging – not part of the training pipeline.)
    """
    stack = []
    for tok in reversed(tokens):
        if tok in _VARIABLES_SET or tok == '<C>':
            stack.append(tok)
        elif tok in PREFIX_BINARY_OPS:
            a = stack.pop() if stack else '?'
            b = stack.pop() if stack else '?'
            _fmt = {'add': f'({a}+{b})', 'sub': f'({a}-{b})',
                    'mul': f'({a}*{b})', 'div': f'({a}/{b})',
                    'pow': f'({a}**{b})'}
            stack.append(_fmt.get(tok, f'{tok}({a},{b})'))
        elif tok in PREFIX_UNARY_OPS:
            a = stack.pop() if stack else '?'
            stack.append(f'{tok}({a})')
        # skip <SOS>, <EOS>, <PAD>, <UNK>
    return stack[0] if stack else ''


# ── Quick tests ───────────────────────────────────────────────────────────────
_tokenization_test_cases = [
    ('sin(x1) + x1**2',       'sin x1'),
    ('x1 * x2 - x1 / x2',     'sub mul'),
    ('exp(-x1**2 / 2)',       'exp'),
    ('sqrt(x1**2 + x2**2)',   'sqrt'),
]

print('\nTokenizer smoke tests:')
print(f'  {"Equation":<35s}  Prefix tokens')
print('  ' + '-' * 65)
# Explicit nested-structure flattening check
nested_example = ['add', ['mul', 'x1', 'x2'], ['sin', 'x1']]
print(f"  Flatten nested example: {nested_example} -> {flatten_prefix(nested_example)}")
for eq, expected_substr in _tokenization_test_cases:
    toks = tokenize_equation(eq)
    ok   = expected_substr in ' '.join(toks)
    ids  = tokens_to_ids(['<SOS>'] + toks + ['<EOS>'])
    back = prefix_tokens_to_infix(toks)
    is_flat = all(not isinstance(t, list) for t in toks)
    status = '✓' if ok and is_flat else '✗'
    print(f'  {status} {eq:<35s}  {' '.join(toks)}')
    print(f'    Flattened: {is_flat}')
    print(f'    IDs: {ids}')
    print(f'    Back to infix: {back}')
    print()


## 4. Real Equation Dataset (Feynman/Bonus)

Training samples are loaded from extracted AI Feynman / Bonus files and converted to prefix token sequences with SymPy.

Each training sample is a tuple `(X_points, y_points, target_token_ids)` where  
- `X_points`, `y_points` form the point cloud for regression  
- `target_token_ids` are the token ids of the masked skeleton equation  


In [ ]:
UNARY_OPS_SYMPY  = ['sin', 'cos', 'exp', 'log', 'sqrt', 'abs', 'tanh']
BINARY_OPS_SYMPY = ['+', '-', '*', '/', '**']


def safe_eval(expr_str: str, x_vals: dict, eps: float = 1e-8):
    """Evaluate a sympy expression string at given variable values."""
    try:
        syms   = {k: sp.Symbol(k) for k in x_vals}
        expr   = sp.sympify(expr_str, locals=syms)
        func   = sp.lambdify(list(syms.values()), expr, modules='numpy')
        vals   = np.array([x_vals[k] for k in syms], dtype=np.float64)
        result = func(*vals)
        result = np.where(np.isfinite(result), result, np.nan)
        return result
    except Exception:
        return None


class EquationGenerator:
    """
    Randomly builds symbolic equations using a recursive parse-tree approach.
    Mirrors the procedure described in SymbolicGPT Section 3.1.
    """

    def __init__(self,
                 max_depth:   int   = 3,
                 n_vars:      int   = 2,
                 const_prob:  float = 0.5,
                 const_range: tuple = (-2.1, 2.1)):
        self.max_depth   = max_depth
        self.n_vars      = n_vars
        self.const_prob  = const_prob
        self.const_range = const_range
        self.var_names   = [f'x{i+1}' for i in range(n_vars)]

    def _random_expr(self, depth: int) -> sp.Expr:
        if depth >= self.max_depth or (depth > 0 and random.random() < 0.3):
            if random.random() < 0.7:
                return sp.Symbol(random.choice(self.var_names))
            else:
                c = random.uniform(*self.const_range)
                return sp.Float(round(c, 2))

        if random.random() < 0.5:          # binary op
            op  = random.choice(BINARY_OPS_SYMPY)
            lhs = self._random_expr(depth + 1)
            rhs = self._random_expr(depth + 1)
            op_map = {
                '+':  lhs + rhs,
                '-':  lhs - rhs,
                '*':  lhs * rhs,
                '/':  lhs / (rhs + sp.Float(1e-6)),
                '**': lhs ** random.choice([2, 3, 0.5, -1]),
            }
            return op_map[op]
        else:                              # unary op
            op  = random.choice(UNARY_OPS_SYMPY)
            arg = self._random_expr(depth + 1)
            op_map = {
                'sin': sp.sin, 'cos': sp.cos, 'exp': sp.exp,
                'log': sp.log, 'sqrt': sp.sqrt, 'abs': sp.Abs,
                'tanh': sp.tanh,
            }
            return op_map[op](arg)

    def _add_constants(self, expr: sp.Expr) -> sp.Expr:
        if random.random() < self.const_prob:
            expr = sp.Float(round(random.uniform(*self.const_range), 2)) * expr
        if random.random() < self.const_prob:
            expr = expr + sp.Float(round(random.uniform(*self.const_range), 2))
        return expr

    def generate(self) -> tuple | None:
        """Returns (expr_str, prefix_tokens) or None if invalid."""
        try:
            expr        = self._random_expr(0)
            expr        = self._add_constants(expr)
            expr        = sp.simplify(expr)
            expr_str    = str(expr)
            prefix_toks = sympy_to_prefix(expr)
            return expr_str, prefix_toks
        except Exception:
            return None


class PointCloudSampler:
    """Sample a point cloud {(x, y)} from a symbolic equation."""

    def __init__(self,
                 n_points:  int   = 50,
                 x_range:   tuple = (-3.0, 3.0),
                 n_vars:    int   = 2,
                 noise_std: float = 0.0):
        self.n_points  = n_points
        self.x_range   = x_range
        self.n_vars    = n_vars
        self.noise_std = noise_std

    def sample(self, expr_str: str) -> tuple:
        """Returns (X_np [N, d], y_np [N]) or (None, None) on failure."""
        var_names = [f'x{i+1}' for i in range(self.n_vars)]
        syms      = {v: sp.Symbol(v) for v in var_names}
        try:
            parsed = sp.sympify(expr_str, locals=syms)
            func   = sp.lambdify(list(syms.values()), parsed, modules='numpy')
        except Exception:
            return None, None

        X = np.random.uniform(*self.x_range,
                              size=(self.n_points, self.n_vars)).astype(np.float32)
        try:
            y = func(*[X[:, i] for i in range(self.n_vars)]).astype(np.float32)
        except Exception:
            return None, None

        if not np.all(np.isfinite(y)):
            return None, None

        if self.noise_std > 0:
            y = y + np.random.normal(0, self.noise_std * np.std(y) + 1e-8, y.shape)
        return X, y


# ── Dataset ───────────────────────────────────────────────────────────────────

class SRDataset(Dataset):
    """
    Symbolic-regression dataset using **prefix-notation** token sequences.

    Each item
    ---------
    'points'   : FloatTensor [N, d+1]    (x1…xd, y) – point cloud
    'input_ids': LongTensor  [T]         <SOS> followed by prefix tokens
    'labels'   : LongTensor  [T]         prefix tokens followed by <EOS>
    'eq_str'   : str                     original infix expression
    'prefix'   : str                     space-joined prefix token sequence
    """

    def __init__(self,
                 size:      int   = 5000,
                 max_eq_len:int   = 80,
                 n_vars:    int   = 2,
                 n_points:  int   = 50,
                 noise_std: float = 0.0,
                 max_depth: int   = 3,
                 x_range:   tuple = (-3.0, 3.0),
                 feynman_catalogue: list | None = None):
        """
        Parameters
        ----------
        feynman_catalogue : required real-data catalogue (Feynman/Bonus entries).
        """
        self.max_eq_len = max_eq_len
        self.n_vars     = n_vars
        self.data       = []

        if not feynman_catalogue:
            raise ValueError('SRDataset requires a non-empty real-data catalogue.')

        eligible = [
            e for e in feynman_catalogue
            if e.get('data_path') and (e.get('n_vars') in (None, n_vars))
        ]
        if not eligible:
            raise ValueError(f'No catalogue entries found for n_vars={n_vars} with data files.')

        rng = np.random.default_rng(42)
        order = rng.permutation(len(eligible)).tolist()

        attempts = 0
        idx = 0
        pbar = tqdm(total=size, desc='Loading real equations')
        while len(self.data) < size and attempts < max(size * 20, len(eligible) * 5):
            attempts += 1
            entry = eligible[order[idx % len(order)]]
            idx += 1

            X, y = load_equation_points(entry, n_points=n_points, rng=rng)
            if X is None:
                continue

            if noise_std > 0:
                y = y + np.random.normal(0, noise_std * np.std(y) + 1e-8, y.shape)

            prefix_toks = equation_str_to_prefix(entry['formula'], n_vars=n_vars)
            before = len(self.data)
            self._store(X, y, entry['formula'], prefix_toks)
            if len(self.data) > before:
                pbar.update(1)

        pbar.close()
        print(f'[SRDataset] Loaded {len(self.data)} real samples (attempts={attempts})')

    def _store(self, X, y, eq_str, prefix_toks):
        """Validate length and append an item to self.data."""
        tokens = ['<SOS>'] + prefix_toks + ['<EOS>']
        if len(tokens) > self.max_eq_len + 2:
            return
        if any(t == '<UNK>' for t in prefix_toks):
            return   # skip equations with unsupported operators
        ids   = tokens_to_ids(tokens)
        cloud = np.concatenate([X, y.reshape(-1, 1)], axis=1).astype(np.float32)
        self.data.append({
            'cloud':  cloud,
            'ids':    ids,
            'eq_str': eq_str,
            'prefix': ' '.join(prefix_toks),
        })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item      = self.data[idx]
        ids       = item['ids']
        input_ids = torch.tensor(ids[:-1], dtype=torch.long)   # SOS … last
        labels    = torch.tensor(ids[1:],  dtype=torch.long)   # first … EOS
        cloud     = torch.tensor(item['cloud'], dtype=torch.float32)
        return cloud, input_ids, labels, item['eq_str'], item['prefix']


def collate_fn(batch):
    clouds, input_ids_list, labels_list, eq_strs, prefixes = zip(*batch)

    max_len = max(x.size(0) for x in input_ids_list)
    input_padded  = torch.stack([
        F.pad(x, (0, max_len - x.size(0)), value=PAD_IDX) for x in input_ids_list
    ])
    labels_padded = torch.stack([
        F.pad(x, (0, max_len - x.size(0)), value=PAD_IDX) for x in labels_list
    ])
    max_pts       = max(c.size(0) for c in clouds)
    clouds_padded = torch.stack([
        F.pad(c, (0, 0, 0, max_pts - c.size(0)), value=0.0) for c in clouds
    ])
    return clouds_padded, input_padded, labels_padded, list(eq_strs), list(prefixes)


# ── Smoke test ────────────────────────────────────────────────────────────────
print('\nGenerating smoke-test dataset (real equations)…')
if not CATALOGUE:
    raise RuntimeError('CATALOGUE is empty. Ensure data/ contains AI Feynman files.')

smoke_ds = SRDataset(size=min(100, len(CATALOGUE)), n_vars=2, n_points=30, max_depth=2,
                     feynman_catalogue=CATALOGUE)
smoke_dl = DataLoader(smoke_ds, batch_size=8, collate_fn=collate_fn)
batch    = next(iter(smoke_dl))
clouds, inp, lbl, eqs, pfx = batch
print(f'Cloud shape  : {clouds.shape}')
print(f'Input shape  : {inp.shape}')
print(f'Label shape  : {lbl.shape}')
print(f'Sample eq    : {eqs[0]}')
print(f'Sample prefix: {pfx[0]}')



## 5. Model Architecture

### 5.1 T-Net – Order-Invariant Point Cloud Encoder

Inspired by PointNet (Qi et al., 2017) and re-interpreted as a **semantic tree encoder** in  
Malik et al. (NeurIPS ML4PS 2025), the T-Net maps a variable-size point cloud  
`X ∈ ℝ^{N×(d+1)}` to a fixed-size embedding `w ∈ ℝ^e` through:

1. Per-point shared MLP layers → `N × 4e`
2. Global max-pooling → `1 × 4e`  (order-invariant)
3. Two FC layers → `1 × e`

In [ ]:
class TNet(nn.Module):
    """
    Order-invariant point-cloud encoder.
    Maps [B, N, d+1] → [B, emb_dim] and supports variable-length point clouds.
    Uses LayerNorm to normalise each point across its feature dimension, which
    avoids dependence on batch-level statistics.
    """

    def __init__(self, in_dim: int, emb_dim: int = 256):
        super().__init__()
        self.input_norm = nn.LayerNorm(in_dim)

        # Shared MLP applied independently to each point
        self.shared_mlp = nn.Sequential(
            nn.Linear(in_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim * 2),
            nn.ReLU(),
            nn.Linear(emb_dim * 2, emb_dim * 4),
            nn.ReLU(),
        )

        # Aggregation MLP
        self.fc1 = nn.Sequential(
            nn.Linear(emb_dim * 4, emb_dim * 2),
            nn.ReLU(),
        )
        self.fc2 = nn.Linear(emb_dim * 2, emb_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, N, d+1]
        x = self.input_norm(x)
        x = self.shared_mlp(x)
        x = torch.max(x, dim=1)[0]

        x = self.fc1(x)
        x = self.fc2(x)
        return x


# Smoke test
tnet   = TNet(in_dim=3, emb_dim=64).to(DEVICE)
x_test = torch.randn(4, 50, 3).to(DEVICE)
emb    = tnet(x_test)
print(f'TNet output: {emb.shape}')   # should be [4, 64]


### 5.2 Transformer Decoder (GPT-style)

A causal (prefix-self-attention) Transformer decoder generates the equation skeleton
token by token.  The point-cloud embedding `w_D` is injected by adding it  
(broadcast) to every position of the token-embedding + positional-encoding matrix,  
following the SymbolicGPT formulation:

```
h = W_p + W_D + X_eq * W_t
```

where `W_D` is `w_D` expanded to match sequence length.

In [ ]:
def generate_causal_mask(seq_len: int) -> torch.Tensor:
    """Create an upper-triangular boolean mask that blocks future positions."""
    return torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()


class CausalSelfAttention(nn.Module):
    """Multi-head causal (masked) self-attention block."""

    def __init__(self, emb_dim: int, n_heads: int, dropout: float = 0.1,
                 max_len: int = 256):
        super().__init__()
        assert emb_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = emb_dim // n_heads

        self.qkv_proj  = nn.Linear(emb_dim, 3 * emb_dim, bias=False)
        self.out_proj  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.register_buffer('causal_mask', generate_causal_mask(max_len), persistent=False)

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        B, T, C = x.shape
        H, D    = self.n_heads, self.head_dim

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(C, dim=-1)

        q = q.view(B, T, H, D).transpose(1, 2)
        k = k.view(B, T, H, D).transpose(1, 2)
        v = v.view(B, T, H, D).transpose(1, 2)

        scale  = D ** -0.5
        scores = (q @ k.transpose(-2, -1)) * scale

        causal_mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        if key_padding_mask is not None:
            scores = scores.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), float('-inf'))

        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)

        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    """Single GPT-style transformer block."""

    def __init__(self, emb_dim: int, n_heads: int,
                 ff_mult: int = 4, dropout: float = 0.1, max_len: int = 256):
        super().__init__()
        self.ln1   = nn.LayerNorm(emb_dim)
        self.attn  = CausalSelfAttention(emb_dim, n_heads, dropout, max_len)
        self.ln2   = nn.LayerNorm(emb_dim)
        self.ff    = nn.Sequential(
            nn.Linear(emb_dim, ff_mult * emb_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        x = x + self.attn(self.ln1(x), key_padding_mask)
        x = x + self.ff(self.ln2(x))
        return x


class SymbolicGPT(nn.Module):
    """
    Full SymbolicGPT model:
        T-Net encoder  +  GPT decoder
    """

    def __init__(self,
                 in_dim: int,
                 vocab_size: int,
                 emb_dim: int = 256,
                 n_heads: int = 8,
                 n_layers: int = 4,
                 max_seq_len: int = 100,
                 dropout: float = 0.1):
        super().__init__()
        self.emb_dim = emb_dim

        self.tnet = TNet(in_dim=in_dim, emb_dim=emb_dim)

        self.tok_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.dataset_proj = nn.Linear(emb_dim, emb_dim)
        self.pos_emb = nn.Embedding(max_seq_len, emb_dim)
        self.emb_drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(emb_dim, n_heads, dropout=dropout, max_len=max_seq_len)
            for _ in range(n_layers)
        ])
        self.ln_f    = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self,
                cloud: torch.Tensor,
                input_ids: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        B, T = input_ids.shape

        dataset_embedding = self.dataset_proj(self.tnet(cloud))

        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        token_embedding = self.tok_emb(input_ids)
        positional_embedding = self.pos_emb(positions)

        combined_embedding = (
            token_embedding
            + dataset_embedding.unsqueeze(1)
            + positional_embedding
        )
        h = self.emb_drop(combined_embedding)

        for block in self.blocks:
            h = block(h, key_padding_mask)

        h = self.ln_f(h)
        return self.lm_head(h)

    @torch.no_grad()
    def generate(self,
                 cloud: torch.Tensor,
                 max_new_tokens: int = 80,
                 temperature: float = 0.8,
                 top_k: int = 40) -> list:
        self.eval()
        device   = next(self.parameters()).device
        ids      = torch.tensor([[SOS_IDX]], dtype=torch.long, device=device)
        generated = []

        for _ in range(max_new_tokens):
            logits = self(cloud, ids)
            logits = logits[:, -1, :] / temperature

            if top_k > 0:
                vals, _ = torch.topk(logits, top_k)
                threshold = vals[:, -1].unsqueeze(-1)
                logits = logits.masked_fill(logits < threshold, float('-inf'))

            probs  = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            ids     = torch.cat([ids, next_id], dim=1)
            generated.append(next_id.item())

            if next_id.item() == EOS_IDX:
                break

        return generated


# ── Smoke test ────────────────────────────────────────────────────────────────
N_VARS = 2
model  = SymbolicGPT(
    in_dim      = N_VARS + 1,
    vocab_size  = VOCAB_SIZE,
    emb_dim     = 128,
    n_heads     = 4,
    n_layers    = 2,
    max_seq_len = 100,
    dropout     = 0.1,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')

cloud_t  = torch.randn(4, 50, N_VARS + 1).to(DEVICE)
inp_t    = torch.randint(0, VOCAB_SIZE, (4, 20)).to(DEVICE)
logits_t = model(cloud_t, inp_t)
print(f'Logits shape: {logits_t.shape}')


## 6. Training

We train SymbolicGPT with **cross-entropy loss** (ignoring `<PAD>` tokens)  
using the **Adam** optimiser with cosine learning-rate decay.

In [ ]:
# ── Hyper-parameters ─────────────────────────────────────────────────────────
CFG = dict(
    # Dataset
    train_size   = 8000,
    val_size     = 1000,
    n_vars       = 2,
    n_points     = 50,
    max_depth    = 3,
    # Model
    emb_dim      = 256,
    n_heads      = 8,
    n_layers     = 4,
    max_seq_len  = 100,
    dropout      = 0.1,
    # Training
    epochs       = 10,
    batch_size   = 64,
    lr           = 3e-4,
    weight_decay = 1e-4,
    grad_clip    = 1.0,
)

print('Configuration:', CFG)

In [ ]:
# ── Build datasets ────────────────────────────────────────────────────────────
if not CATALOGUE:
    raise RuntimeError('No local equation catalogue found. Populate data/ before training.')

usable_catalogue = [e for e in CATALOGUE if e.get('data_path')]
if not usable_catalogue:
    raise RuntimeError('Catalogue has no linked equation data files.')

rng = np.random.default_rng(42)
perm = rng.permutation(len(usable_catalogue))
split = max(1, int(0.8 * len(usable_catalogue)))
train_catalogue = [usable_catalogue[i] for i in perm[:split]]
val_catalogue   = [usable_catalogue[i] for i in perm[split:]] or train_catalogue

print('Building training set…')
train_ds = SRDataset(size=CFG['train_size'], n_vars=CFG['n_vars'],
                     n_points=CFG['n_points'], max_depth=CFG['max_depth'],
                     feynman_catalogue=train_catalogue)
print('Building validation set…')
val_ds   = SRDataset(size=CFG['val_size'], n_vars=CFG['n_vars'],
                     n_points=CFG['n_points'], max_depth=CFG['max_depth'],
                     feynman_catalogue=val_catalogue)

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'],
                      shuffle=True, collate_fn=collate_fn, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'],
                      shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f'Train batches: {len(train_dl)} | Val batches: {len(val_dl)}')



In [ ]:
# ── Instantiate model & optimiser ─────────────────────────────────────────────
model = SymbolicGPT(
    in_dim      = CFG['n_vars'] + 1,
    vocab_size  = VOCAB_SIZE,
    emb_dim     = CFG['emb_dim'],
    n_heads     = CFG['n_heads'],
    n_layers    = CFG['n_layers'],
    max_seq_len = CFG['max_seq_len'],
    dropout     = CFG['dropout'],
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay'],
)

total_steps = CFG['epochs'] * len(train_dl)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=1e-6
)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

print(f'Model params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────

def token_accuracy(logits, labels, pad_idx=PAD_IDX):
    """Fraction of non-PAD tokens predicted correctly."""
    preds  = logits.argmax(-1)                 # [B, T]
    mask   = labels != pad_idx
    correct = (preds == labels) & mask
    return correct.sum().item() / max(mask.sum().item(), 1)


def run_epoch(model, loader, optimizer=None, grad_clip=None, desc='Train'):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss, total_acc, n_batches = 0.0, 0.0, 0
    pbar = tqdm(loader, desc=desc, leave=False)

    for clouds, inp, lbl, _, _ in pbar:
        clouds = clouds.to(DEVICE)
        inp    = inp.to(DEVICE)
        lbl    = lbl.to(DEVICE)
        pad_mask = (inp == PAD_IDX)           # True where padded

        with torch.set_grad_enabled(is_train):
            logits = model(clouds, inp, key_padding_mask=pad_mask)  # [B, T, V]
            B, T, V = logits.shape
            loss    = criterion(logits.view(B * T, V), lbl.view(B * T))
            acc     = token_accuracy(logits, lbl)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            if grad_clip:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            scheduler.step()

        total_loss += loss.item()
        total_acc  += acc
        n_batches  += 1
        pbar.set_postfix(loss=f'{total_loss/n_batches:.4f}',
                         acc=f'{total_acc/n_batches:.4f}')

    return total_loss / n_batches, total_acc / n_batches


history = {'train_loss': [], 'val_loss': [],
           'train_acc':  [], 'val_acc':  []}

best_val_loss = float('inf')
best_state    = None

print(f'Training for {CFG["epochs"]} epochs on {DEVICE}…')
for epoch in range(1, CFG['epochs'] + 1):
    t0 = time.time()

    tr_loss, tr_acc = run_epoch(
        model, train_dl, optimizer,
        grad_clip=CFG['grad_clip'], desc=f'Epoch {epoch} Train')

    vl_loss, vl_acc = run_epoch(
        model, val_dl, optimizer=None, desc=f'Epoch {epoch} Val')

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    elapsed = time.time() - t0
    print(f'Epoch {epoch:2d}/{CFG["epochs"]} | '
          f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | '
          f'val_loss={vl_loss:.4f} val_acc={vl_acc:.4f} | '
          f'{elapsed:.1f}s')

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_state    = copy.deepcopy(model.state_dict())
        print(f'  ✓ New best model saved (val_loss={best_val_loss:.4f})')

# Restore best weights
model.load_state_dict(best_state)
print('\nTraining complete. Best val_loss:', best_val_loss)

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, CFG['epochs'] + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Train', marker='o')
axes[0].plot(epochs_range, history['val_loss'],   label='Val',   marker='s')
axes[0].set_title('Cross-Entropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs_range, history['train_acc'], label='Train', marker='o')
axes[1].plot(epochs_range, history['val_acc'],   label='Val',   marker='s')
axes[1].set_title('Token Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()

## 7. BFGS Constant Optimisation

After the GPT decoder generates a **skeleton equation** (with `<C>` placeholders),
we optimise the constant values using the **Broyden–Fletcher–Goldfarb–Shanno (BFGS)**  
algorithm, minimising the MSE on the input point cloud.

This decouples structural learning (model) from numerical fitting (BFGS).

In [ ]:
def skeleton_to_callable(skeleton: str, n_vars: int):
    """
    Replace <C> placeholders with sympy symbols C0, C1, … and
    return a callable f(X, consts) → numpy array.
    """
    var_syms   = [sp.Symbol(f'x{i+1}') for i in range(n_vars)]
    c_count    = skeleton.count('<C>')
    const_syms = [sp.Symbol(f'C{i}') for i in range(c_count)]

    # Replace each <C> with its symbol
    filled = skeleton
    for i, cs in enumerate(const_syms):
        filled = filled.replace('<C>', str(cs), 1)

    try:
        expr = sp.sympify(filled, locals={str(s): s for s in var_syms + const_syms})
    except Exception:
        return None, 0

    all_syms = var_syms + const_syms
    func     = sp.lambdify(all_syms, expr, modules='numpy')

    def evaluate(X: np.ndarray, consts: np.ndarray) -> np.ndarray:
        # X: [N, n_vars],  consts: [c_count]
        args = [X[:, i] for i in range(n_vars)] + list(consts)
        try:
            result = func(*args)
            if np.isscalar(result):
                result = np.full(len(X), result)
            return np.where(np.isfinite(result), result, 1e10)
        except Exception:
            return np.full(len(X), 1e10)

    return evaluate, c_count


def bfgs_fit_constants(skeleton: str,
                       X: np.ndarray,
                       y: np.ndarray,
                       n_vars: int,
                       n_restarts: int = 5) -> tuple:
    """
    Optimise the constants in a skeleton equation using BFGS.

    Returns
    -------
    best_consts : np.ndarray  (optimised constant values)
    best_mse   : float
    """
    evaluate, c_count = skeleton_to_callable(skeleton, n_vars)
    if evaluate is None or c_count == 0:
        return np.array([]), np.inf

    def mse_objective(consts):
        y_pred = evaluate(X, consts)
        resid  = y_pred - y
        return np.mean(resid ** 2)

    best_consts = np.zeros(c_count)
    best_mse    = np.inf

    for _ in range(n_restarts):
        x0 = np.random.uniform(-3.0, 3.0, c_count)
        try:
            res = minimize(mse_objective, x0, method='BFGS',
                           options={'maxiter': 1000, 'gtol': 1e-8})
            if res.fun < best_mse:
                best_mse    = res.fun
                best_consts = res.x
        except Exception:
            continue

    return best_consts, best_mse


def fill_skeleton(skeleton: str, consts: np.ndarray) -> str:
    """Replace <C> tokens with their optimised float values."""
    result = skeleton
    for c in consts:
        result = result.replace('<C>', f'{c:.4f}', 1)
    return result


# ── Test BFGS ─────────────────────────────────────────────────────────────────
skeleton_test = 'sin(<C> * x1) + <C>'
X_test = np.random.uniform(-3, 3, (50, 1))
y_test = np.sin(1.5 * X_test[:, 0]) + 0.8
consts, mse = bfgs_fit_constants(skeleton_test, X_test, y_test, n_vars=1)
print(f'Skeleton : {skeleton_test}')
print(f'Fitted   : {fill_skeleton(skeleton_test, consts)}')
print(f'True     : sin(1.5*x1) + 0.8')
print(f'MSE      : {mse:.6f}')

## 8. Evaluation Pipeline

We report four metrics following the NeurIPS ML4PS 2025 paper:

| Metric | Description |
|---|---|
| **R²** | Coefficient of determination on hold-out points |
| **RMSE** | Root mean-squared error |
| **Token Accuracy** | % of equation tokens predicted correctly |
| **Exact Match %** | % of equations that are symbolically identical to ground truth |

In [ ]:
def r2_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2) + 1e-10
    return float(1.0 - ss_res / ss_tot)


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def normalised_mse(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-6) -> float:
    """Normalised MSE as defined in SymbolicGPT (eq. 2)."""
    norm  = np.linalg.norm(y_true + eps) ** 2 + 1e-10
    return float(np.sum((y_true - y_pred) ** 2) / norm)


def filter_special_tokens(tokens: list[str]) -> list[str]:
    """Remove non-expression control tokens from generated sequences."""
    return [t for t in tokens if t not in ('<SOS>', '<EOS>', '<PAD>')]


def symbolic_match(pred_str: str, true_str: str) -> bool:
    """Check symbolic equivalence using sympy."""
    try:
        syms  = {f'x{i+1}': sp.Symbol(f'x{i+1}') for i in range(5)}
        expr1 = sp.sympify(pred_str, locals=syms)
        expr2 = sp.sympify(true_str, locals=syms)
        diff  = sp.simplify(expr1 - expr2)
        return diff == 0
    except Exception:
        return False


@torch.no_grad()
def evaluate_model(model, dataset, n_samples=200,
                   n_vars=2, bfgs=True, noise_std=0.0):
    """
    Full evaluation: generate prefix skeleton → convert to infix → BFGS → metrics.

    Returns a dict with aggregated metrics.
    """
    model.eval()
    sampler = PointCloudSampler(
        n_points=100, x_range=(-5.0, -3.0),  # out-of-distribution x range
        n_vars=n_vars, noise_std=noise_std
    )

    results = []
    indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))

    for idx in tqdm(indices, desc='Evaluating'):
        item   = dataset.data[idx]
        eq_str = item['eq_str']
        prefix = item['prefix']          # ground-truth prefix token string

        # Use test-time point cloud (out-of-distribution)
        X_test, y_test = sampler.sample(eq_str)
        if X_test is None:
            continue

        cloud_t = torch.tensor(
            np.concatenate([X_test, y_test.reshape(-1, 1)], axis=1)[np.newaxis],
            dtype=torch.float32, device=DEVICE
        )

        # ── Generate prefix token sequence ────────────────────────────────────
        gen_ids      = model.generate(cloud_t, max_new_tokens=80, top_k=40)
        gen_tokens   = ids_to_tokens(gen_ids)
        gen_pfx_toks = filter_special_tokens(gen_tokens)
        gen_prefix   = ' '.join(gen_pfx_toks)   # for display / logging

        # ── Convert prefix → infix for BFGS and sympy operations ─────────────
        gen_infix = prefix_tokens_to_infix(gen_pfx_toks)

        # ── Token accuracy vs ground-truth prefix token sequence ──────────────
        true_tokens = prefix.split()
        pred_tokens = gen_pfx_toks
        n_match  = sum(p == g for p, g in zip(pred_tokens, true_tokens))
        n_total  = max(len(true_tokens), 1)
        tok_acc  = n_match / n_total

        # ── BFGS constant fitting on the infix skeleton ───────────────────────
        if bfgs and '<C>' in gen_infix:
            consts, _ = bfgs_fit_constants(
                gen_infix, X_test, y_test, n_vars=n_vars
            )
            pred_eq = fill_skeleton(gen_infix, consts)
        else:
            pred_eq = gen_infix.replace('<C>', '1.0')

        # ── Compute predictions ────────────────────────────────────────────────
        try:
            evaluate_fn, _ = skeleton_to_callable(pred_eq, n_vars)
            if evaluate_fn is not None:
                y_pred = evaluate_fn(X_test, np.array([]))
                r2   = r2_score(y_test, y_pred)
                rms  = rmse(y_test, y_pred)
                nmse = normalised_mse(y_test, y_pred)
            else:
                r2 = rms = nmse = float('nan')
        except Exception:
            r2 = rms = nmse = float('nan')

        exact = symbolic_match(pred_eq, eq_str)

        results.append(dict(
            eq_str        = eq_str,
            prefix        = prefix,
            pred_prefix   = gen_prefix,
            pred_eq       = pred_eq,
            r2=r2, rmse=rms, nmse=nmse,
            tok_acc=tok_acc, exact=exact,
        ))

    # ── Aggregate ─────────────────────────────────────────────────────────────
    finite = [r for r in results if np.isfinite(r['r2'])]
    agg = {
        'n_total'       : len(results),
        'n_valid'       : len(finite),
        'r2_mean'       : np.nanmean([r['r2']      for r in finite]),
        'rmse_mean'     : np.nanmean([r['rmse']     for r in finite]),
        'nmse_mean'     : np.nanmean([r['nmse']     for r in finite]),
        'token_acc_mean': np.nanmean([r['tok_acc']  for r in results]),
        'exact_match'   : np.mean([r['exact']       for r in results]),
        'details'       : results,
    }
    return agg


print('Evaluating on validation set…')
eval_results = evaluate_model(model, val_ds, n_samples=200, n_vars=CFG['n_vars'])

print('\n' + '='*55)
print('         EVALUATION RESULTS (validation set)')
print('='*55)
print(f"  Samples evaluated : {eval_results['n_total']}")
print(f"  Valid predictions : {eval_results['n_valid']}")
print(f"  R²  (mean)        : {eval_results['r2_mean']:.4f}")
print(f"  RMSE (mean)       : {eval_results['rmse_mean']:.4f}")
print(f"  NMSE (mean)       : {eval_results['nmse_mean']:.4f}")
print(f"  Token Accuracy    : {eval_results['token_acc_mean']*100:.2f}%")
print(f"  Exact Match       : {eval_results['exact_match']*100:.2f}%")
print('='*55)


## 9. Qualitative Results – Prediction Examples

### Decoding Strategy

Greedy decoding:
- Stable but limited exploration

Top-k sampling:
- Generates diverse candidate equations

We use both for better performance.

### Baseline Comparison

As a simple baseline, we also fit a linear regression model on the same `(X, y)` pairs.
This helps contrast symbolic model behavior with a purely linear predictor.


In [ ]:
details = eval_results['details']

print('\nSample predictions (first 5):')
print('-' * 80)
for r in details[:5]:
    print(f"True equation    : {r['eq_str']}")
    print(f"True prefix      : {r['prefix']}")
    print(f"Pred prefix      : {r['pred_prefix']}")
    print(f"Pred equation    : {r['pred_eq']}")
    print(f"R²={r['r2']:.4f}  RMSE={r['rmse']:.4f}  "          f"TokAcc={r['tok_acc']*100:.1f}%  Exact={r['exact']}")
    print('-' * 80)


In [ ]:
# ── Visualise 1-D predictions ─────────────────────────────────────────────────
# Filter to 1-variable predictions for easy plotting
# (retrain or use the saved model if n_vars=1 was trained)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

plot_count = 0
for r in details:
    if plot_count >= 6:
        break
    if not np.isfinite(r.get('r2', float('nan'))):
        continue

    eq_str   = r['eq_str']
    pred_eq  = r['pred_eq']
    n_v      = CFG['n_vars']

    x_plot = np.linspace(-4, 4, 200)
    X_plot = np.column_stack([x_plot] + [np.zeros_like(x_plot)] * (n_v - 1))

    try:
        var_syms = {f'x{i+1}': sp.Symbol(f'x{i+1}') for i in range(n_v)}
        true_fn  = sp.lambdify(list(var_syms.values()),
                               sp.sympify(eq_str, locals=var_syms), 'numpy')
        y_true   = true_fn(*[X_plot[:, i] for i in range(n_v)])

        pred_fn  = sp.lambdify(list(var_syms.values()),
                               sp.sympify(pred_eq, locals=var_syms), 'numpy')
        y_pred   = pred_fn(*[X_plot[:, i] for i in range(n_v)])

        if not (np.all(np.isfinite(y_true)) and np.all(np.isfinite(y_pred))):
            continue

        ax = axes[plot_count]
        ax.plot(x_plot, y_true, 'b-',  linewidth=2,   label='True')
        ax.plot(x_plot, y_pred, 'r--', linewidth=1.5, label='Predicted')
        ax.set_ylim(np.percentile(y_true, 1) - 1, np.percentile(y_true, 99) + 1)
        ax.set_title(f"R²={r['r2']:.3f}", fontsize=9)
        ax.set_xlabel('x1')
        ax.legend(fontsize=7)
        plot_count += 1
    except Exception:
        continue

for i in range(plot_count, 6):
    axes[i].axis('off')

plt.suptitle('True (blue) vs Predicted (red) equations', fontsize=12)
plt.tight_layout()
plt.savefig('predictions.png', dpi=120)
plt.show()

## 10. Noise-Robustness Experiment

Following the PIGP / SymbolicDPO paper (NeurIPS ML4PS 2024), we measure how  
performance degrades under Gaussian noise added to the target `y` values.

In [ ]:
noise_levels  = [0.0, 0.05, 0.1, 0.2, 0.3]
noise_results = {}

for σ in noise_levels:
    print(f'\nEvaluating noise_std={σ}…')

    # Build a small noisy dataset for evaluation
    noisy_ds = SRDataset(size=500, n_vars=CFG['n_vars'],
                         n_points=CFG['n_points'],
                         max_depth=CFG['max_depth'],
                         noise_std=σ)

    res = evaluate_model(model, noisy_ds, n_samples=100,
                         n_vars=CFG['n_vars'], noise_std=σ)
    noise_results[σ] = res
    print(f'  R²={res["r2_mean"]:.4f}  '
          f'RMSE={res["rmse_mean"]:.4f}  '
          f'TokAcc={res["token_acc_mean"]*100:.2f}%  '
          f'Exact={res["exact_match"]*100:.2f}%')

# ── Summary table ─────────────────────────────────────────────────────────────
print('\n' + '='*65)
print(f'{"Noise σ":>10} | {"R²":>8} | {"RMSE":>8} | {"TokAcc%":>9} | {"Exact%":>7}')
print('-'*65)
for σ, res in noise_results.items():
    print(f'{σ:>10.2f} | '
          f'{res["r2_mean"]:>8.4f} | '
          f'{res["rmse_mean"]:>8.4f} | '
          f'{res["token_acc_mean"]*100:>9.2f} | '
          f'{res["exact_match"]*100:>7.2f}')
print('='*65)

In [ ]:
# ── Plot noise robustness ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sigma_vals = list(noise_results.keys())
r2_vals    = [noise_results[s]['r2_mean']            for s in sigma_vals]
rmse_vals  = [noise_results[s]['rmse_mean']           for s in sigma_vals]
tok_vals   = [noise_results[s]['token_acc_mean']*100  for s in sigma_vals]

axes[0].plot(sigma_vals, r2_vals,   'bo-', linewidth=2)
axes[0].set_xlabel('Noise σ (fraction of std)')
axes[0].set_ylabel('R²')
axes[0].set_title('R² vs Noise Level')
axes[0].set_ylim(0, 1.05)

axes[1].plot(sigma_vals, rmse_vals, 'rs-', linewidth=2)
axes[1].set_xlabel('Noise σ (fraction of std)')
axes[1].set_ylabel('RMSE')
axes[1].set_title('RMSE vs Noise Level')

axes[2].plot(sigma_vals, tok_vals,  'g^-', linewidth=2)
axes[2].set_xlabel('Noise σ (fraction of std)')
axes[2].set_ylabel('Token Accuracy (%)')
axes[2].set_title('Token Accuracy vs Noise Level')

plt.tight_layout()
plt.savefig('noise_robustness.png', dpi=120)
plt.show()

## 11. AI Feynman Dataset Evaluation (Optional)

The **AI Feynman** dataset (Udrescu & Tegmark, 2020) contains 100 physics equations  
from undergraduate physics.  Below we load it from a public URL, parse a subset,  
and run our model on it.  If the download fails, the section is gracefully skipped.

In [ ]:
FEYNMAN_EQUATIONS = [
    # (name, expression in terms of x1, x2, ...)
    ('I.6.2a',   'exp(-x1**2/2)/sqrt(2*3.14159)'),
    ('I.12.5',   'x1**2 * x2'),
    ('I.18.14',  'x1 * x2 * x3 * sin(x4)'),
    ('I.39.1',   '1.5 * x1 * x2'),
    ('I.43.16',  'x1 * x2 * x3 / x4'),
    ('I.43.31',  'x1 * x2 * x3'),
    ('II.4.23',  'x1 / (4 * 3.14159 * x2 * x3)'),
    ('II.21.32', 'x1 / (4 * 3.14159 * x2 * x3 * (1 - x4))'),
    ('II.35.21', 'x1 * x2 * tanh(x2 * x3 / (x4 * x5))'),
    ('II.38.3',  'x1 * x2 * x3 / x4'),
]

print(f'AI Feynman subset: {len(FEYNMAN_EQUATIONS)} equations')

feynman_rows = []
for name, expr in FEYNMAN_EQUATIONS:
    # Determine actual variable count
    used_vars = len([v for v in ['x1', 'x2', 'x3', 'x4', 'x5'] if v in expr])
    sampler_i = PointCloudSampler(n_points=100, x_range=(0.5, 5.0),
                                  n_vars=used_vars, noise_std=0.0)
    X_f, y_f = sampler_i.sample(expr)
    if X_f is None:
        print(f'  SKIP {name} (failed to sample)')
        continue

    # Use our model (trained on 2-variable data) – zero-pad extra vars
    n_v_model = CFG['n_vars']
    X_padded  = np.zeros((len(X_f), n_v_model), dtype=np.float32)
    X_padded[:, :min(used_vars, n_v_model)] = X_f[:, :min(used_vars, n_v_model)]

    cloud_t = torch.tensor(
        np.concatenate([X_padded, y_f.reshape(-1, 1)], axis=1)[np.newaxis],
        dtype=torch.float32, device=DEVICE
    )

    # ── Generate prefix tokens → convert to infix ────────────────────────────
    gen_ids      = model.generate(cloud_t, max_new_tokens=80, top_k=40)
    gen_tokens   = ids_to_tokens(gen_ids)
    gen_pfx_toks = filter_special_tokens(gen_tokens)
    gen_infix    = prefix_tokens_to_infix(gen_pfx_toks)

    # ── BFGS constant fitting ────────────────────────────────────────────────
    if '<C>' in gen_infix:
        consts, mse = bfgs_fit_constants(
            gen_infix, X_padded, y_f, n_vars=n_v_model
        )
        pred_eq = fill_skeleton(gen_infix, consts)
    else:
        pred_eq = gen_infix.replace('<C>', '1.0')
        mse = np.inf

    # ── Evaluate ─────────────────────────────────────────────────────────────
    try:
        evaluate_fn, _ = skeleton_to_callable(pred_eq, n_v_model)
        if evaluate_fn is not None:
            y_pred = evaluate_fn(X_padded, np.array([]))
            r2_f   = r2_score(y_f, y_pred)
            rmse_f = rmse(y_f, y_pred)
        else:
            r2_f = rmse_f = float('nan')
    except Exception:
        r2_f = rmse_f = float('nan')

    feynman_rows.append({
        'name':     name,
        'true_eq':  expr,
        'pred_pfx': ' '.join(gen_pfx_toks),
        'pred_eq':  pred_eq,
        'r2':       r2_f,
        'rmse':     rmse_f,
    })

print('\nAI Feynman Results:')
print(f'{"Name":>12} | {"R²":>8} | {"RMSE":>10} | Predicted')
print('-' * 80)
for row in feynman_rows:
    print(f"{row['name']:>12} | {row['r2']:>8.4f} | "
          f"{row['rmse']:>10.4f} | {row['pred_eq'][:45]}")

r2_mean_f   = np.nanmean([r['r2']   for r in feynman_rows])
rmse_mean_f = np.nanmean([r['rmse'] for r in feynman_rows])
print('-' * 80)
print(f'  Mean R² = {r2_mean_f:.4f}   Mean RMSE = {rmse_mean_f:.4f}')


## 12. R² Distribution Visualisation

In [ ]:
r2_vals = [r['r2'] for r in eval_results['details'] if np.isfinite(r['r2'])]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of R²
axes[0].hist(r2_vals, bins=30, edgecolor='black', color='steelblue', alpha=0.8)
axes[0].axvline(np.mean(r2_vals), color='red', linestyle='--',
                linewidth=2, label=f'Mean = {np.mean(r2_vals):.3f}')
axes[0].set_xlabel('R²')
axes[0].set_ylabel('Count')
axes[0].set_title('R² Distribution (Validation Set)')
axes[0].legend()

# Cumulative distribution (as in SymbolicGPT Figure 2)
nmse_vals = np.array([r['nmse'] for r in eval_results['details']
                      if np.isfinite(r.get('nmse', float('nan')))])
log_nmse  = np.log10(nmse_vals + 1e-20)
sorted_ln = np.sort(log_nmse)
cumulative = np.arange(1, len(sorted_ln) + 1) / len(sorted_ln) * 100

axes[1].plot(sorted_ln, cumulative, 'b-', linewidth=2, label='SymbolicGPT (ours)')
axes[1].set_xlabel('Log₁₀(Normalised MSE)')
axes[1].set_ylabel('Proportion (%)')
axes[1].set_title('Cumulative Log NMSE Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('r2_distribution.png', dpi=120)
plt.show()

## 13. Final Results Summary Table

In [ ]:
print('\n' + '='*70)
print('            FINAL RESULTS SUMMARY')
print('='*70)
print(f'  Architecture : SymbolicGPT (T-Net + GPT Decoder + BFGS)')
print(f'  Training data: {CFG["train_size"]} real Feynman/Bonus samples')
print(f'  Variables    : {CFG["n_vars"]}  |  Points per sample : {CFG["n_points"]}')
print(f'  Embedding dim: {CFG["emb_dim"]}  |  Layers: {CFG["n_layers"]}  '
      f'|  Heads: {CFG["n_heads"]}')
print()
print('  Validation-set metrics:')
print(f"    R² (mean)        : {eval_results['r2_mean']:.4f}")
print(f"    RMSE (mean)      : {eval_results['rmse_mean']:.4f}")
print(f"    Token Accuracy   : {eval_results['token_acc_mean']*100:.2f}%")
print(f"    Exact Match      : {eval_results['exact_match']*100:.2f}%")
print()
print('  Comparison (from NeurIPS ML4PS 2025, Feynman dataset):')
print(f'    AI Feynman (2020)  Exact: ~100%  R² ≈ 1.00  (heuristic graph)')
print(f'    LASR (2024)        Exact:  72%   R² ≈ 0.99  (concept library)')
print(f'    QDSR (2024)        Exact:  91.6% R² ≈ 0.99  (discrete search)')
print(f'    Malik et al. 2025  Exact:  19.6% R² ≈ 0.976 (neural baseline)')
print(f'    Ours (this run)    Exact: {eval_results["exact_match"]*100:5.1f}% '
      f'R² ≈ {eval_results["r2_mean"]:.3f}  (SymbolicGPT reimplementation)')
print('='*70)

## 14. Save & Load Model Checkpoint

In [ ]:
checkpoint_path = 'symbolic_gpt_checkpoint.pt'

torch.save({
    'model_state_dict': model.state_dict(),
    'config':           CFG,
    'vocab':            VOCAB,
    'history':          history,
}, checkpoint_path)
print(f'Model saved to {checkpoint_path}')


def load_model(path: str):
    ckpt = torch.load(path, map_location=DEVICE)
    cfg  = ckpt['config']
    m    = SymbolicGPT(
        in_dim      = cfg['n_vars'] + 1,
        vocab_size  = VOCAB_SIZE,
        emb_dim     = cfg['emb_dim'],
        n_heads     = cfg['n_heads'],
        n_layers    = cfg['n_layers'],
        max_seq_len = cfg['max_seq_len'],
        dropout     = cfg['dropout'],
    ).to(DEVICE)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    return m


# Verify reload
loaded_model = load_model(checkpoint_path)
print('Model reloaded successfully.')

## 15. Interactive Inference

Given a new point cloud `(X, y)`, run inference to discover the equation.

In [ ]:
def tokens_to_sympy(tokens: list[str]) -> sp.Expr | None:
    """Convert prefix tokens to a SymPy expression via infix reconstruction."""
    infix = prefix_tokens_to_infix(tokens)
    local_syms = {f'x{i}': sp.Symbol(f'x{i}') for i in range(1, 6)}
    try:
        return sp.sympify(infix, locals=local_syms)
    except Exception:
        return None


def optimize_constants(equation, X: np.ndarray, y: np.ndarray, n_vars: int, n_restarts: int = 5) -> str:
    """Fit <C> placeholders with BFGS and return the final equation string."""
    skeleton = str(equation) if equation is not None else ''
    if not skeleton:
        return ''
    if '<C>' not in skeleton:
        return skeleton

    consts, _ = bfgs_fit_constants(skeleton, X, y, n_vars=n_vars, n_restarts=n_restarts)
    return fill_skeleton(skeleton, consts)


def compute_mse(eq, X: np.ndarray, y: np.ndarray, n_vars: int) -> float:
    """Compute equation MSE against targets; return inf on invalid evaluation."""
    try:
        eq_str = str(eq)
        eval_fn, _ = skeleton_to_callable(eq_str, n_vars)
        if eval_fn is None:
            return float('inf')
        y_pred = eval_fn(X, np.array([]))
        return float(np.mean((y_pred - y) ** 2))
    except Exception:
        return float('inf')


def mutate(eq_tokens: list[str]) -> list[str]:
    """Simple token-level mutation for hybrid neural-symbolic search."""
    mutated = list(eq_tokens)
    if len(mutated) > 2:
        idx = random.randint(1, len(mutated) - 1)
        mutation_ops = [op for op in PREFIX_BINARY_OPS if op in VOCAB]
        if mutation_ops:
            mutated[idx] = random.choice(mutation_ops)
    return mutated


def baseline_model(X: np.ndarray, y: np.ndarray) -> float:
    """Simple linear baseline (R² on training points)."""
    if HAS_SKLEARN:
        model = LinearRegression()
        model.fit(X, y)
        return float(model.score(X, y))

    # Fallback when scikit-learn is unavailable
    X_aug = np.concatenate([X, np.ones((len(X), 1), dtype=X.dtype)], axis=1)
    coef, *_ = np.linalg.lstsq(X_aug, y, rcond=None)
    y_pred = X_aug @ coef
    return r2_score(y, y_pred)


def generate_candidates(model, cloud_t: torch.Tensor, num_samples: int = 5,
                        temperature: float = 0.7, top_k: int = 40) -> list[list[str]]:
    """Generate token candidates (string tokens) via greedy + top-k + mutation."""
    candidates = []

    # Greedy candidate: stable decoding
    greedy_ids = model.generate(cloud_t, temperature=1.0, top_k=1)
    greedy_tokens = mutate(filter_special_tokens(ids_to_tokens(greedy_ids)))
    candidates.append(greedy_tokens)

    # Top-k sampled candidates: diverse exploration
    for _ in range(max(0, num_samples - 1)):
        eq_ids = model.generate(cloud_t, temperature=temperature, top_k=top_k)
        eq_tokens = mutate(filter_special_tokens(ids_to_tokens(eq_ids)))
        candidates.append(eq_tokens)

    return candidates


def select_best(candidates: list[str], X: np.ndarray, y: np.ndarray, n_vars: int) -> tuple[str | None, float]:
    """Select the candidate equation with minimum MSE on (X, y)."""
    best = None
    best_loss = float('inf')

    for eq in candidates:
        loss = compute_mse(eq, X, y, n_vars=n_vars)
        if loss < best_loss:
            best = eq
            best_loss = loss

    return best, best_loss


def full_inference_pipeline(model, X: np.ndarray, y: np.ndarray,
                            n_vars: int, num_samples: int = 5,
                            temperature: float = 0.7, top_k: int = 40,
                            n_bfgs_restarts: int = 5) -> dict:
    """Run generation + selection + BFGS and return a structured result dict."""
    cloud_np = np.concatenate([X, y.reshape(-1, 1)], axis=1).astype(np.float32)
    cloud_t  = torch.tensor(cloud_np[np.newaxis], dtype=torch.float32, device=DEVICE)

    candidates = generate_candidates(
        model, cloud_t, num_samples=num_samples,
        temperature=temperature, top_k=top_k,
    )

    candidate_eqs = []
    for toks in candidates:
        equation = tokens_to_sympy(toks)
        candidate_eqs.append(str(equation) if equation is not None else prefix_tokens_to_infix(toks))

    best_eq, best_mse = select_best(candidate_eqs, X, y, n_vars=n_vars)
    final_eq = optimize_constants(best_eq, X, y, n_vars=n_vars, n_restarts=n_bfgs_restarts)

    print(f'Best pre-BFGS candidate MSE: {best_mse:.6f}')
    return {
        'candidates': candidate_eqs,
        'best_pre_bfgs': best_eq,
        'best_pre_bfgs_mse': best_mse,
        'final_equation': final_eq,
    }


def symbolic_regression_infer(X: np.ndarray,
                               y: np.ndarray,
                               model,
                               n_vars: int,
                               temperature: float = 0.7,
                               top_k: int = 40,
                               n_bfgs_restarts: int = 5,
                               num_samples: int = 5) -> str:
    """
    Full inference pipeline:
      tokens → symbolic expression → optimisation → final equation
    """
    result = full_inference_pipeline(
        model, X, y,
        n_vars=n_vars,
        num_samples=num_samples,
        temperature=temperature,
        top_k=top_k,
        n_bfgs_restarts=n_bfgs_restarts,
    )
    print(f"Final equation: {result['final_equation']}")
    return result['final_equation']


# ── End-to-end demo: input dataset → prediction → BFGS → comparison ─────────
print('\n── End-to-end demo ───────────────────────────────────────────────')
X_demo = np.random.uniform(-3, 3, (200, CFG['n_vars'])).astype(np.float32)
y_demo = (np.sin(X_demo[:, 0]) + np.cos(X_demo[:, 1])).astype(np.float32)

print('Input dataset sample (first 5 rows):')
print(np.round(np.concatenate([X_demo[:5], y_demo[:5, None]], axis=1), 4))

demo_result = full_inference_pipeline(
    model, X_demo, y_demo,
    n_vars=CFG['n_vars'],
    num_samples=5,
    temperature=0.7,
    top_k=40,
    n_bfgs_restarts=5,
)

pred_pre = demo_result['best_pre_bfgs']
pred_final = demo_result['final_equation']
print(f"\nPredicted equation (pre-BFGS): {pred_pre}")
print(f"Predicted equation (post-BFGS): {pred_final}")

# Quick evaluation + baseline
for label, eq_str in [('Pre-BFGS', pred_pre), ('Post-BFGS', pred_final)]:
    eval_fn, _ = skeleton_to_callable(eq_str, CFG['n_vars'])
    if eval_fn is not None:
        y_hat = eval_fn(X_demo, np.array([]))
        print(f"{label:<10s} | R²={r2_score(y_demo, y_hat):.4f}  RMSE={rmse(y_demo, y_hat):.4f}")

baseline_r2 = baseline_model(X_demo, y_demo)
print(f"Linear baseline | R²={baseline_r2:.4f}")
print('Ground truth    : sin(x1) + cos(x2)')


## 16. Architecture Diagram

The figure below summarises the full SymbolicGPT pipeline.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

boxes = [
    (0.04, 0.3, 0.14, 0.4, 'Point Cloud\n{(x,y)}ᵢ\n[N × (d+1)]',  'lightblue'),
    (0.23, 0.2, 0.14, 0.6, 'T-Net\n(Order-invariant\nEncoder)',      'lightyellow'),
    (0.42, 0.3, 0.14, 0.4, 'Embedding\nw_D ∈ ℝᵉ',                  'lightgreen'),
    (0.60, 0.1, 0.16, 0.8, 'GPT Decoder\n(8 Transformer\nBlocks)',   'lightsalmon'),
    (0.81, 0.2, 0.14, 0.6, 'Skeleton\ne.g. sin(<C>*x1)+<C>',        'plum'),
]

for x, y, w, h, label, color in boxes:
    rect = matplotlib.patches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle='round,pad=0.02',
        facecolor=color, edgecolor='gray', linewidth=1.5,
        transform=ax.transAxes
    )
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label,
            ha='center', va='center', fontsize=8, transform=ax.transAxes)

# Arrows
arrow_props = dict(arrowstyle='->', lw=1.5, color='black')
for x_start, x_end in [(0.18, 0.23), (0.37, 0.42), (0.56, 0.60), (0.76, 0.81)]:
    ax.annotate('', xy=(x_end, 0.5), xytext=(x_start, 0.5),
                xycoords='axes fraction', arrowprops=arrow_props)

# BFGS box below skeleton
rect2 = matplotlib.patches.FancyBboxPatch(
    (0.81, -0.12), 0.14, 0.28,
    boxstyle='round,pad=0.02',
    facecolor='wheat', edgecolor='gray', linewidth=1.5,
    transform=ax.transAxes
)
ax.add_patch(rect2)
ax.text(0.88, 0.04, 'BFGS\nConstant\nOptimisation',
        ha='center', va='center', fontsize=8, transform=ax.transAxes)
ax.annotate('', xy=(0.88, 0.2), xytext=(0.88, -0.02),
            xycoords='axes fraction',
            arrowprops=dict(arrowstyle='<->', lw=1.5, color='darkblue'))

ax.set_title('SymbolicGPT Architecture', fontsize=13, pad=10)
plt.tight_layout()
plt.savefig('architecture_diagram.png', dpi=120)
plt.show()

## 17. Conclusions

This notebook implements **SymbolicGPT** for symbolic regression, combining:

| Component | Role |
|---|---|
| **T-Net** | Order-invariant encoder of input point clouds |
| **GPT Decoder** | Auto-regressive equation skeleton generator |
| **`<C>` masking** | Decouples structural learning from constant fitting |
| **BFGS optimisation** | Post-hoc constant refinement |

### Failure Cases

Observed issues:
- Correct structure but wrong constants
- Missing terms
- Incorrect exponent placement

Insight:
Model achieves strong numerical accuracy but struggles with exact symbolic recovery.

### Key Insights

- Neural models capture local structure effectively
- Global symbolic correctness remains challenging
- Separating structure learning and constant optimization improves performance

### Final Pipeline

Dataset → Tokenization → T-Net → Transformer → Skeleton Equation → BFGS → Final Equation

### Future directions

* Integrate **PIGP / SymbolicDPO** (Jha et al., 2024) to refine skeletons with GP.
* Add **learned concept libraries** (Malik et al., 2025) for recurring sub-expressions.
* Scale to the full **AI Feynman** benchmark (100 equations, multi-variable).
* Apply **sparse attention** (sliding window) for longer equation sequences.
* Explore **curriculum learning** – start simple, grow complexity.
